# Figure 1: convergence of the sampler

The collapsed log joint $\log p(x, z \mid \alpha, \beta, \gamma, K)$ of every chain, one panel per $K$, over the whole run: the dashed line marks where $\alpha$ starts being sampled and the dotted line the end of burn-in.  The horizontal axis is logarithmic so the burn-in is visible.  These are convergence diagnostics, not model fit: values are not comparable across $K$.

Input: `RESULTS/runs/*/` (`blat run`, one chain each).

**Where to start Jupyter.**  Start it from the top directory (`jupyter lab`), so the file
browser shows the notebooks and the results side by side.  Starting it anywhere else
works too: Jupyter always runs a notebook in the notebook's own folder, and the first
cell finds the top directory from there.

**Which results.**  Set `RESULTS` in the first cell to the results directory of your run,
relative to the top directory: `example/results-quick` (the default) or
`example/results-thorough`.  When `run_final.sh` executes the notebooks it sets it for
you, through the environment variable `BLAT_RESULTS`.

In [ ]:
import os, sys
from pathlib import Path

# the top directory: the nearest folder, from here upwards, that holds the blat package
TOP = next(p for p in [Path.cwd(), *Path.cwd().parents]
                      if (p / "blat" / "__init__.py").exists())
sys.path.insert(0, str(TOP))

# the results to draw, relative to the top directory (or absolute)
RESULTS = TOP / os.environ.get("BLAT_RESULTS", "example/results-quick")
FIGURES = RESULTS / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
print("results:", RESULTS)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from blat import figures as F

def save(fig, name):
    for suffix in ("pdf", "png"):
        fig.savefig(FIGURES / f"{name}.{suffix}", dpi=300, bbox_inches="tight")
    print("wrote", FIGURES / f"{name}.pdf")

In [ ]:
by_k = F.load_runs(RESULTS / "runs")
{k: len(v) for k, v in by_k.items()}

Agreement between chains at each $K$: split $\hat R$ of the log joint and of $\sum_k\alpha_k$ over the sampling phase, the spread of the chains' mean log joint, and the worst matched cosine between any chain's types and the medoid chain's.

In [ ]:
F.cross_chain_table(by_k).round(3)

In [ ]:
fig = F.plot_traces(by_k)
save(fig, "figure1-convergence")

Warnings raised by each chain (`warnings` in its summary.yaml):

In [ ]:
for k, runs in by_k.items():
    for r in runs:
        for w in r.record["warnings"]:
            print(f"K={k} {r.name}: {w}")